In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [10]:
import torch
import tiktoken
from utility import token_ids_to_text, text_to_token_ids
from model.architecture import generate_text_simple, GPTModel
from model.dummy_model import load_yaml

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
config = load_yaml("../model/config/model_config.yaml")["model"]
model = GPTModel(config)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=config["context_length"]
)
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

Output text: Every effort moves you investments cradle redirect WisdomeezCTardoooky ReynoldsLayout


## Calculating text generation loss

In [24]:
input1 = "every effort moves"
input2 = "I really like"
enc1 = torch.tensor(tokenizer.encode(input1))
enc2 = torch.tensor(tokenizer.encode(input2))
inputs = torch.stack((enc1, enc2))
print(inputs)
print(inputs.shape)

target1 = " effort moves you"
target2 = " really like chocolate"
t_enc1 = torch.tensor(tokenizer.encode(target1))
t_enc2 = torch.tensor(tokenizer.encode(target2))
targets = torch.stack((t_enc1, t_enc2))
print(targets)
print(targets.shape)

tensor([[16833,  3626,  6100],
        [   40,  1107,   588]])
torch.Size([2, 3])
tensor([[ 3626,  6100,   345],
        [ 1107,   588, 11311]])
torch.Size([2, 3])


In [26]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1)
print(probas.shape)


torch.Size([2, 3, 50257])


In [27]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print(token_ids)
print(token_ids.shape)

tensor([[[38771],
         [33791],
         [42733]],

        [[ 2216],
         [20971],
         [47764]]])
torch.Size([2, 3, 1])


In [28]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")


Targets batch 1:  effort moves you
Outputs batch 1:  queens condom Waves


In [ ]:
# print the initial softmax scores of the 2 sets of 3 tokens
text_idx = 0
target_probas1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"Text 1 scores: {target_probas1}")

text_idx = 1
target_probas2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print(f"Text 2 scores: {target_probas2}")

Text 1 scores: tensor([9.3011e-06, 1.3143e-05, 9.6696e-06])
Text 1 scores: tensor([9.4056e-06, 2.1140e-05, 9.1337e-06])


In [34]:
# to get probability scores, 6 steps:
# logits -> probabilities -> target probabilities -> log probabilities -> avg log probability -> negative avg probabilit

# step 4:
log_probas = torch.log(torch.cat((target_probas1, target_probas2)))
print(log_probas)

tensor([-11.5854, -11.2396, -11.5465, -11.5742, -10.7643, -11.6035])


In [35]:
# step 5: average log prob
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-11.3856)


In [38]:
# step 6: turn neg (cross entropy loss)
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(11.3856)


In [40]:
# for cross entropy loss we need to flatten the tokens along the batch
# this means:
# logits dim  [2, 3, 50257] ---> [6, 50257]
# targets dim [2, 3] ----------> [6]

logits_flat = logits.flatten(0, 1)
print(logits_flat.shape)
targets_flat = targets.flatten(0)
print(targets.shape)

torch.Size([6, 50257])
torch.Size([2, 3])


In [41]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(11.3856)


In [42]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(88044.4609)


In [ ]:
# the above perplexity means that the model is uncertain about which among number of <perplexity val> 
# tokens to generate